# s99 MDD template map (Step 2b)

READ ONLY. Maps every heading of the clean MDD template (`config.mdd.template`, from Step 2a) to what it asks for, the matching spec sections (`config.mdd.spec`), the evidence sheets of the Step 1 evidence workbooks (`<doc_output_root>/<output_folder>/evidence/mdd_evidence_<key>.xlsx`), the required human inputs (headings of `config.mdd.human_inputs`) and the previous MDD profile (`previous_mdd_profile.xlsx`: classification + excerpt). Each section gets a status COVERED / PARTIAL / GAP with what is missing; spec content that has no place in the template gets a proposed destination. No document text is written. The only write is `<doc_output_root>/<output_folder>/template_map.xlsx` (meta, map, spec_unplaced, evidence_unplaced, template structure, inputs, checks, run_summary). No MLflow run and no `reports/run_summaries/` file: the request restricts writes to this workbook (same as s99_mdd_evidence_pack); the run summary is a sheet.

In [ ]:
# The project has no requirements.lock yet (same note as s99_mdd_clean_template); this module needs python-docx + openpyxl + pyyaml.
%pip install python-docx==1.2.0 openpyxl pyyaml --quiet
dbutils.library.restartPython()

In [ ]:
# s99_mdd_template_map: READ ONLY. The only write is <doc_output_root>/<output_folder>/template_map.xlsx (config.mdd).
import sys, os, re, json, hashlib, importlib, random, tempfile, shutil
from datetime import datetime, timezone
from collections import Counter
sys.path.insert(0, os.getcwd())            # notebook lives in src/
import common
importlib.reload(common)
import numpy as np, pandas as pd, yaml, docx, openpyxl
from docx.oxml.ns import qn

MODULE = "s99_mdd_template_map"
PROJECT_ROOT = os.path.dirname(os.getcwd())
CFG_PATH = os.path.join(PROJECT_ROOT, "config", "config.yaml")
with open(CFG_PATH) as f:
    CFG_TEXT = f.read()
cfg = yaml.safe_load(CFG_TEXT)
SEED = cfg["random_seed"]
random.seed(SEED); np.random.seed(SEED)
MDD = cfg["mdd"]
RUN_TIME = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def git_info():
    """Branch / head commit of the Git folder via the Repos API; 'unknown' if not available."""
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        oid = w.workspace.get_status(PROJECT_ROOT.replace("/Workspace", "", 1)).object_id
        r = w.repos.get(oid)
        return r.head_commit_id or "unknown", r.branch or "unknown"
    except Exception:
        return "unknown", "unknown"

CFG_SHA = sha256_file(CFG_PATH)
GIT, BRANCH = git_info()

def mdd_duplicate_keys(text):
    """Keys repeated inside the mdd: block (yaml keeps the LAST one). Recorded as a note; config is human-maintained."""
    m = re.search(r"^mdd:\s*$", text, re.M)
    block = text[m.end():]
    end = re.search(r"^\S", block, re.M)
    block = block[:end.start()] if end else block
    keys = re.findall(r"^  ([A-Za-z_]\w*):", block, re.M)
    return sorted(k for k, n in Counter(keys).items() if n > 1)

MDD_DUP_KEYS = mdd_duplicate_keys(CFG_TEXT)

def resolve(rel, bases):
    for b in bases:
        p = os.path.join(b, rel)
        if os.path.exists(p):
            return p
    return None

HOME = os.path.dirname(PROJECT_ROOT)            # config.mdd.spec is relative to the user's home (.assistant/skills/...)
TPL = resolve(MDD["template"], [PROJECT_ROOT])
SPEC = resolve(MDD["spec"], [PROJECT_ROOT, HOME])
HI = resolve(MDD["human_inputs"], [PROJECT_ROOT])
MDDR = os.path.join(MDD["doc_output_root"], MDD["output_folder"])
EVID = os.path.join(MDDR, "evidence")
PROFILE = os.path.join(MDDR, "previous_mdd_profile.xlsx")      # written by s99_mdd_clean_template (Step 2a)
OUT = os.path.join(MDDR, "template_map.xlsx")
MODELS = MDD["models"]
EV_WB = {m["key"]: os.path.join(EVID, f"mdd_evidence_{m['key']}.xlsx") for m in MODELS}
for name, p in [("template", TPL), ("spec", SPEC), ("human_inputs", HI), ("profile", PROFILE)] + [(f"evidence {k}", v) for k, v in EV_WB.items()]:
    assert p and os.path.exists(p), f"{name} not found: {p}"

WATCH = sorted({MDDR, os.path.join(PROJECT_ROOT, "docs"), os.path.join(PROJECT_ROOT, "config"), os.path.dirname(SPEC)})
def snapshot(roots):
    """Content hash of every file (size/mtime of /Workspace files are not stable between stat calls)."""
    out = {}
    for r in roots:
        for dp, dn, fn in os.walk(r):
            for f in fn:
                p = os.path.join(dp, f)
                out[p] = sha256_file(p)
    return out
SNAP_BEFORE = snapshot(WATCH)
INPUT_SHA = {"config": CFG_SHA, "template": sha256_file(TPL), "spec": sha256_file(SPEC), "human_inputs": sha256_file(HI),
             "profile": sha256_file(PROFILE), **{f"evidence_{k}": sha256_file(v) for k, v in EV_WB.items()}}

print(f"Module   : {MODULE} | run {RUN_TIME} UTC | branch {BRANCH} | git {GIT[:12]} | config {CFG_SHA[:12]}")
print(f"Template : {TPL}\nSpec     : {SPEC}\nInputs   : {HI}\nProfile  : {PROFILE}")
for k, v in EV_WB.items():
    print(f"Evidence : {k} {v}")
print(f"Output   : {OUT} (exists before run: {os.path.exists(OUT)})")
print(f"config.mdd duplicate keys (yaml keeps the last): {MDD_DUP_KEYS or 'none'} -> template used = {MDD['template']}")
print(f"Read-only snapshot: {len(SNAP_BEFORE)} files under {WATCH}")

In [ ]:
tdoc = docx.Document(TPL)
STYLE_NAME = {s.style_id: s.name for s in tdoc.styles}

def local(el):
    return el.tag.split("}")[-1] if isinstance(el.tag, str) else ""

def el_text(el):
    return "".join(t.text or "" for t in el.iter(qn("w:t")))

def norm(s):
    return re.sub(r"\s+", " ", str(s or "").replace("\xa0", " ")).strip()

def p_style(el):
    ps = el.find(f"{qn('w:pPr')}/{qn('w:pStyle')}")
    sid = ps.get(qn("w:val")) if ps is not None else "Normal"
    return STYLE_NAME.get(sid, sid)

def has_toc(el):
    instr = "".join(t.text or "" for t in el.iter(qn("w:instrText")))
    instr += " ".join(f.get(qn("w:instr")) or "" for f in el.iter(qn("w:fldSimple")))
    return "TOC" in instr

def tbl_info(el):
    trs = el.findall(qn("w:tr"))
    grid = [[norm(el_text(tc)) for tc in tr.findall(qn("w:tc"))] for tr in trs]
    ts = el.find(f"{qn('w:tblPr')}/{qn('w:tblStyle')}")
    style = STYLE_NAME.get(ts.get(qn("w:val")), ts.get(qn("w:val"))) if ts is not None else None
    labels = " ; ".join(" ".join(c for c in r if c) for r in grid[1:] if any(r))   # FORM tables keep their item labels
    return {"rows": len(trs), "cols": max((len(r) for r in grid), default=0), "style": style,
            "header": " | ".join(grid[0]) if grid else "", "labels": labels}

HEADINGS, TABLES, COVER = [], [], []
stack, cur = {}, None
for pos, el in enumerate(tdoc.element.body.iterchildren()):
    tag = local(el)
    if tag == "p":
        name, txt = p_style(el), norm(el_text(el))
        m = re.match(r"^heading (\d)$", name.lower())
        if m:
            lvl = int(m.group(1))
            parent = next((stack[l] for l in range(lvl - 1, 0, -1) if l in stack), "")
            stack = {k: v for k, v in stack.items() if k < lvl}
            stack[lvl] = txt
            cur = {"order": len(HEADINGS) + 1, "level": lvl, "heading": txt, "style": name, "parent": parent, "body_pos": pos,
                   "placeholders": [], "instructions": [], "tables": [], "toc_field": False}
            HEADINGS.append(cur)
            continue
        if cur is not None and has_toc(el):
            cur["toc_field"] = True
        if txt:
            if cur is None:
                COVER.append({"body_pos": pos, "style": name, "text": txt})
            elif txt.startswith("[Section content:"):
                cur["placeholders"].append(txt)
            else:
                cur["instructions"].append(txt)
    elif tag == "tbl":
        info = {"table_no": len(TABLES), "section_order": cur["order"] if cur else None,
                "section": cur["heading"] if cur else "(cover)", **tbl_info(el)}
        TABLES.append(info)
        if cur is not None:
            cur["tables"].append(info["table_no"])
    elif tag == "sdt" and cur is not None and has_toc(el):
        cur["toc_field"] = True

# styles used: every paragraph in the body (incl. table cells), table styles, headers/footers
STYLE_USE = Counter()
for p in tdoc.element.body.iter(qn("w:p")):
    STYLE_USE[("body paragraph", p_style(p))] += 1
for t in TABLES:
    STYLE_USE[("table", t["style"] or "(none)")] += 1
for i, s in enumerate(tdoc.sections):
    for kind, part in (("header", s.header), ("footer", s.footer), ("first_page_header", s.first_page_header)):
        if part.is_linked_to_previous:
            continue
        for p in part._element.iter(qn("w:p")):
            STYLE_USE[(kind, p_style(p))] += 1
TPL_HEAD_DF = pd.DataFrame([{**{k: h[k] for k in ("order", "level", "heading", "parent", "style", "toc_field")},
                             "placeholder": "\n".join(h["placeholders"]), "instruction_text": "\n".join(h["instructions"]),
                             "tables": ", ".join(map(str, h["tables"]))} for h in HEADINGS])
TPL_TBL_DF = pd.DataFrame(TABLES)
TPL_STYLE_DF = pd.DataFrame([{"used_in": k[0], "style": k[1], "count": v} for k, v in sorted(STYLE_USE.items())])
print(f"{len(HEADINGS)} headings | levels {dict(Counter(h['level'] for h in HEADINGS))} | {len(TABLES)} tables | "
      f"cover text {[c['text'] for c in COVER]}")
print(f"headings with placeholder: {sum(bool(h['placeholders']) for h in HEADINGS)} | with other text: "
      f"{sum(bool(h['instructions']) for h in HEADINGS)} | TOC field under: {[h['heading'] for h in HEADINGS if h['toc_field']]}")
print("styles:", "; ".join(f"{k[0]}:{k[1]}={v}" for k, v in sorted(STYLE_USE.items())))
display(TPL_HEAD_DF[["order", "level", "heading", "tables"]])

In [ ]:
def md_headings(text):
    """Markdown headings outside fenced code blocks (config/code comments start with '#')."""
    out, in_code = [], False
    for ln, line in enumerate(text.splitlines(), 1):
        if line.strip().startswith("```"):
            in_code = not in_code
            continue
        if in_code:
            continue
        m = re.match(r"^(#{1,6})\s+(.+?)\s*#*\s*$", line)
        if m:
            out.append({"line": ln, "level": len(m.group(1)), "title": m.group(2).strip()})
    return out

def spec_id(t):
    for pat in (r"^Appendix ([A-Z])\b", r"^([A-Z]\.\d+(?:\.\d+)*)\s", r"^(\d+(?:\.\d+)*)\.?\s"):
        m = re.match(pat, t)
        if m:
            return m.group(1)
    return None

with open(SPEC) as f:
    SPEC_TEXT = f.read()
SPEC_H = md_headings(SPEC_TEXT)
for s in SPEC_H:
    s["id"] = spec_id(s["title"])
    s["parent_id"] = s["id"].rsplit(".", 1)[0] if s["id"] and "." in s["id"] else None
SPEC_IDS = [s["id"] for s in SPEC_H if s["id"]]
assert len(SPEC_IDS) == len(set(SPEC_IDS)), f"duplicate spec section ids: {[i for i, n in Counter(SPEC_IDS).items() if n > 1]}"
SPEC_BY = {s["id"]: s for s in SPEC_H if s["id"]}
for s in SPEC_H:
    s["has_children"] = any(x["parent_id"] == s["id"] for x in SPEC_H) if s["id"] else False
SPEC_DF = pd.DataFrame(SPEC_H)

# human inputs: level-2 headings and the placeholders still open under each
with open(HI) as f:
    HI_TEXT = f.read()
HI_LINES, HI_H = HI_TEXT.splitlines(), md_headings(HI_TEXT)
HI_SECS = []
for i, h in enumerate(HI_H):
    if h["level"] != 2:
        continue
    nxt = next((x["line"] for x in HI_H[i + 1:] if x["level"] <= 2), len(HI_LINES) + 1)
    body = "\n".join(HI_LINES[h["line"]:nxt - 1])
    angles = re.findall(r"<[^<>\n]*>", body)
    rest = re.sub(r"<[^<>\n]*>", "", body)
    ph = angles + re.findall(r"\bTBC\b", rest) + [m.strip() for m in re.findall(r"[^\n:]*:\s*null\b", rest)]
    HI_SECS.append({"heading": h["title"], "line": h["line"], "n_chars": len(body.strip()), "open_placeholders": ph})
HI_BY = {h["heading"]: h for h in HI_SECS}
HI_DF = pd.DataFrame([{**h, "open_placeholders": "\n".join(h["open_placeholders"])} for h in HI_SECS])

# evidence workbooks (Step 1): sheet names, copy status, gaps, final feature list
EV = {}
for k, p in EV_WB.items():
    wb = openpyxl.load_workbook(p, read_only=True, data_only=True)
    idx = list(wb["index"].iter_rows(values_only=True))
    si, st = idx[0].index("sheet"), idx[0].index("status")
    g = list(wb["gaps"].iter_rows(values_only=True))
    fl = list(wb["L5_final_list"].iter_rows(values_only=True)) if "L5_final_list" in wb.sheetnames else []
    fh = next((i for i, r in enumerate(fl) if r and r[0] == "feature"), None)
    EV[k] = {"sheets": list(wb.sheetnames), "status": {r[si]: r[st] for r in idx[1:] if r[si]},
             "gaps": [dict(zip(g[0], r)) for r in g[1:] if any(v is not None for v in r)],
             "final_features": [r[0] for r in fl[fh + 1:] if r and r[0]] if fh is not None else []}
    wb.close()

def sheet_ok(s, k):
    return s in EV[k]["sheets"] and EV[k]["status"].get(s, "COPIED") not in ("EMPTY", "MISSING")

ALL_SHEETS = sorted(set().union(*[set(v["sheets"]) for v in EV.values()]))
EV_DF = pd.DataFrame([{"sheet": s, **{f"in_{k}": s in EV[k]["sheets"] for k in EV},
                       **{f"status_{k}": (EV[k]["status"].get(s, "structural") if s in EV[k]["sheets"] else "absent") for k in EV}}
                      for s in ALL_SHEETS])

# previous-MDD profile (Step 2a): classification + excerpt per section, same order as the template
PROF = pd.read_excel(PROFILE, sheet_name="sections")
PROF_OK = len(PROF) == len(HEADINGS) and all(
    int(r.level) == h["level"] and norm(r.heading) == h["heading"] for r, h in zip(PROF.itertuples(), HEADINGS))

print(f"spec: {len(SPEC_H)} headings, {len(SPEC_IDS)} numbered sections | human inputs: {[h['heading'] for h in HI_SECS]}")
print("human inputs with open placeholders:", {h["heading"]: len(h["open_placeholders"]) for h in HI_SECS})
for k in EV:
    print(f"evidence {k}: {len(EV[k]['sheets'])} sheets, {sum(not sheet_ok(s, k) for s in EV[k]['sheets'])} empty, "
          f"{len(EV[k]['gaps'])} gaps, {len(EV[k]['final_features'])} final features")
print(f"profile: {len(PROF)} sections, headings match template (level, text, order): {PROF_OK} | "
      f"classification {dict(Counter(PROF['classification']))}")

In [ ]:
# One rule per template heading (matched on level + heading text; the per-variable H3s under 'Model Variable binning'
# share one rule). spec = spec section ids; ev = evidence sheet names ('a|b' = either); hi = mdd_inputs.md headings;
# missing = content the template asks for that no spec section / evidence sheet supplies; gap=True -> GAP regardless;
# gap_types = gap types from the evidence 'gaps' sheet that are reported for this section.
DEC, DEV_, OPEN, ROLES, PURP = ("Decisions and approvals (date, who, evidence)", "Deviations from the plan (spec v1.0.5)",
                                "Open items", "Roles", "Purpose and intended use")
def R(key, level, asks, spec=(), ev=(), hi=(), missing=(), gap=False, kind="CONTENT", gap_types=()):
    return dict(key=key, level=level, asks=asks, spec=list(spec), ev=list(ev), hi=list(hi), missing=list(missing),
                gap=gap, kind=kind, gap_types=list(gap_types))

MAP_RULES = [
    R("revision history", 1, "Version history table (date, version, description, author, reviewer)", ["15"], [], [ROLES],
      ["revision rows (dates, versions, authors, reviewers): only config.mdd.doc_version exists; not in docs/mdd_inputs.md"], kind="DOC_CONTROL"),
    R("table of contents", 1, "TOC field (refreshes on open)", kind="AUTO"),
    R("introduction", 1, "Purpose, scope and business background of the scorecard", ["0.1", "1.1", "1.3"], ["registry_fields"], [PURP]),
    R("model inventory information", 2, "Inventory form (item labels in the table column)", ["13.5", "14", "F.3"], ["registry_fields"], [ROLES, PURP],
      ["Model ID, Model Materiality, Model Management Committee and Modeling Data Owner are not in the spec, registry or mdd_inputs.md"]),
    R("key dates of model life-cycle", 2, "Life-cycle stage / completion-date table", ["3.1", "7.7"],
      ["registry_fields", "selection_override", "oot_run_summary"], [DEC],
      ["validation, approval and go-live dates not recorded (E16-E20 not run)"]),
    R("model development process", 1, "Narrative of the end-to-end development steps", ["3.1", "6.0", "E"], ["fun_by_layer"]),
    R("data overview", 1, "Chapter intro (sub-sections only)", ["1.2", "2.1"], ["data_counts_by_window"]),
    R("data received", 2, "Data-source table: data, file name, time duration, no. of files, description", ["1.2", "2.4"],
      ["data_counts_by_window", "registry_fields"], [],
      ["no file/table-level source inventory (source tables and snapshot dates per source)"], gap_types=["NOT_IN_REGISTRY"]),
    R("data processing", 2, "Intro (sub-sections only)", ["2.3", "4.1"], ["data_filter_waterfall"]),
    R("aggregation of files", 3, "How source files are joined / aggregated into the modelling table", ["4.1", "3.3"], [], [],
      ["no report describes how AECB / FTS / CASA / application sources are joined into acard_master"]),
    R("data quality and observations", 3, "Data-quality table (data type, remarks, solution) and chart", ["6.1"],
      ["L0_counts_by_reason", "L0_counts_by_source", "L0_feature_detail"]),
    R("data analysis", 1, "Customers-by-quarter table", ["1.2", "2.1"], ["data_counts_by_window", "target_vintage_monthly"], [],
      ["counts by quarter not tabulated (window counts and monthly vintage only)"]),
    R("performance definition", 2, "Intro (sub-sections only)", ["2.2"], ["target_vintage_monthly"], [DEC]),
    R("roll rate analysis: delinquency severity", 3, "Roll-forward / backward / stationary tables and charts supporting the bad definition",
      ["2.2"], [], [DEC], ["no roll-rate report: target is precomputed (spec 2.2) and fixed by the Policy Team; the spec has no roll-rate step"], gap=True),
    R("vintage analysis: length of assessment period", 3, "Bad capture rate by months on book justifying the performance window",
      ["2.1"], ["target_vintage_monthly"], [], ["capture-rate-by-MOB table not produced (monthly vintage bad rates only)"]),
    R("performance definition", 3, "Good / bad / indeterminate definitions", ["2.2", "2.1"], ["data_status_by_window"], [DEC]),
    R("sample selection", 2, "Sample table: disbursals, accounts, % accounts, bad rate", ["1.2", "5.1"],
      ["data_counts_by_window", "data_status_by_window", "data_confirmed_counts_cfg"]),
    R("exclusion rules", 2, "Exclusion tables (count, %, bad rate, stability)", ["2.3", "D.8"],
      ["data_filter_waterfall", "data_active_reporting_xtab"], [DEC], ["stability of exclusions over time (third template table) not reported"]),
    R("segmentation", 2, "Segmentation analysis: segment counts / bad rates for Dev, OOT, Ultra OOT; choice of segments", ["G", "2.3"],
      ["data_filter_waterfall", "oot_segments", "benchmark_step5_segments"], [],
      ["spec builds one thin-file model (no segmentation study); no Ultra-OOT sample"]),
    R("variable creation", 2, "How candidate variables were derived", ["2.4", "2.5"], ["L0_counts_by_source", "fun_by_source"], [],
      ["variable derivation logic / feature dictionary not in the evidence workbooks"]),
    R("model development and validation", 1, "Intro and data-sufficiency table (information type, sufficiency, issues)", ["6.0", "7.1"],
      ["fun_by_layer"], [], ["data-sufficiency table not produced"]),
    R("split data into training and validation datasets", 2, "Sample-split table (period, % customers, accounts, goods, bads, bad rate)",
      ["5.1", "5.2", "5.5"], ["data_counts_by_window", "data_dev_early_late", "data_tfold_stats"]),
    R("data quality", 2, "Variable-level data quality (missing, special values)", ["6.1"], ["L0_feature_detail", "L0_counts_by_reason"]),
    R("information value", 2, "IV screening and thresholds", ["6.2"], ["L1_iv_table", "L1_counts_by_reason"]),
    R("correlation analysis", 2, "Correlation screening and removals", ["6.5"], ["L4_corr_drops", "L4_pool_summary"]),
    R("trends and multi collinearity", 2, "Trend (risk-relationship stability over time) and VIF", ["6.3", "6.5"],
      ["L2_periods", "L2_period_results", "L4_vif"]),
    R("fine tuning", 2, "Tuning steps and final parameters", ["7.2", "7.5"],
      ["mod_tuning_choice", "mod_fixed_params", "mod_C_shortlist", "mod_C_tfold", "mod_chosen_params", "mod_lgb_rounds"]),
    R("scaling", 2, "Probability-to-score scaling (PDO, odds, points)", ["9.1", "9.3", "D.7"],
      ["cal_platt", "cal_platt_file", "cal_scaling_A", "cal_factor_offset_A"], [DEC],
      ["score-scaling sheets are for Model A only (cal_scaling_A, cal_factor_offset_A); Model C score mapping not in the evidence"]),
    R("model kpis", 2, "Score-tranche table: Dev / Validation / OOT population and bad rates, reverse cumulative bad rate", ["8.2"],
      ["oot_bands", "oot_decile_csv", "mod_oof_metrics"], [],
      ["no separate validation sample (spec uses DEV cross-validation OOF); DEV tranche shares only as dev_share in oot_bands"]),
    R("cc a score thick bureau segment", 1, "Model chapter for the thick-bureau segment", ["G", "2.3"], [], [],
      ["thick-file segment is out of scope (THIN FILE only); this H1 holds the model chapter and must be re-titled for nofts_c / fts_c"], gap=True),
    R("dev", 2, "Dev / Val split table for the segment", ["5.1", "5.5"], ["data_counts_by_window", "data_tfold_stats"], [],
      ["repeats 'Split data into training and validation datasets'; the spec has no Dev/Val hold-out (5-fold CV inside DEV)"]),
    R("summary of variable reduction", 2, "Funnel table: stage, removed, expert inclusion / exclusion, remaining", ["6.0", "6.8"],
      ["fun_by_layer", "fun_drop_rules", "fun_by_layer_source", "L3_psi_summary", "L3_psi_decisions_file"]),
    R("final model variables", 2, "Final variable list with description and trait", ["6.6"], ["L5_final_list", "mod_C_importance"], [],
      ["variable descriptions (feature dictionary) not in the evidence workbooks"]),
    R("model parameters & equation", 2, "Logistic coefficient table (estimate, std error, Wald, p-value, IV) and equation", ["7.3", "7.5"],
      ["mod_chosen_params", "mod_C_importance", "mod_A_coefficients"], [],
      ["coefficient / Wald table fits logistic regression only; Model C (monotonic LightGBM) has parameters, monotone constraints and importance instead"]),
    R("model variable binning", 2, "Intro (one sub-section per variable)", ["7.3", "6.2", "6.7"], ["L1_iv_table", "oot_L6_summary"]),
    R("model performance", 2, "Intro", ["8.2"], ["oot_summary", "oot_metrics"]),
    R("model metrics", 3, "KPI table (Gini, KS, AUC)", ["8.2", "8.3"], ["oot_metrics", "mod_oof_metrics", "oot_delong", "oot_paired_bootstrap"]),
    R("risk tranches", 3, "Score-tranche table: Dev / OOT population, bad rate, capture, SI", ["8.2", "9.3"],
      ["oot_bands", "oot_band_monotonic", "oot_score_psi", "cut_score_bands_C"]),
    R("model results kpi", 3, "Primary / secondary metric tables and charts (ROC, KS, lift, calibration)", ["8.2", "9.1"],
      ["oot_calibration", "oot_calibration_by_decile", "oot_score_psi", "benchmark_step3_deciles", "benchmark_step6_roc_curve", "benchmark_step6_capture_curve"], [],
      ["charts must be drawn from curve data; PNGs exist only for the benchmark common sample (reports/benchmark)"]),
    R("challenger models", 2, "Champion vs challenger variables and performance tables", ["7.1", "8.1", "8.3"],
      ["oot_rules_R1_R4", "oot_metrics", "oot_paired_bootstrap", "fts_cmp_performance", "fts_cmp_paired_bootstrap",
       "benchmark_step1_discrimination", "benchmark_step2_paired_deltas"]),
    R("rationale for choosing final model as champion model", 3, "Why the champion was chosen", ["8.1"],
      ["selection_override", "oot_rules_R1_R4", "cut_C_vs_A"], [DEC, DEV_], gap_types=["SELECTION_NOT_FINAL", "EVIDENCE_REF_MISSING"]),
    R("modelling assumptions , limitations & weaknesses", 2, "Assumptions, limitations and weaknesses", ["15.1", "1.3", "1.4", "8.5"],
      ["gaps", "oot_diagnosis"], [DEV_, OPEN]),
    R("annexure", 1, "Dataset-name table, people and roles, model-name form, review-comment tables", ["15", "16.2", "B"],
      ["inventory_reports", "index", "registry_fields"], [ROLES], ["review / comment tables are completed by the reviewers"]),
]
VARBIN = R("(variable under model variable binning)", 3, "Per-variable bin table: Dev / OOT / Ultra-OOT population, bad rates, SI", ["6.2", "6.7"],
           ["L1_iv_table", "oot_L6_summary", "mod_A_points"], [],
           ["per-bin Dev/OOT population and bad-rate table not in the evidence (oot_L6_summary is per feature, mod_A_points is Model A only); no Ultra-OOT sample"])

def hkey(s):
    return norm(s).lower().rstrip(":").strip()

def match(h):
    if h["level"] == 3 and hkey(h["parent"]) == "model variable binning":
        return VARBIN
    c = [r for r in MAP_RULES if r["level"] == h["level"] and hkey(h["heading"]) == r["key"]]
    if not c:
        c = [r for r in MAP_RULES if r["level"] == h["level"] and hkey(h["heading"]).startswith(r["key"])]
    assert len(c) == 1, f"heading {h['order']} {h['heading']!r}: {len(c)} matching rules"
    return c[0]

RULE_USE, ROWS = Counter(), []
for h, p in zip(HEADINGS, PROF.itertuples()):
    r = match(h)
    RULE_USE[id(r)] += 1
    miss = list(r["missing"])
    if r is VARBIN:
        var = hkey(h["heading"]).split(" ")[0]
        inf = {k: var in {f.lower() for f in EV[k]["final_features"]} for k in EV}
        miss.insert(0, f"heading names previous-model variable '{var}' (in final list: " + ", ".join(f"{k} {'yes' if v else 'no'}" for k, v in inf.items())
                    + "); sub-sections must follow this model's final features (L5_final_list)")
    ev_disp, ev_any = [], False
    for s in r["ev"]:
        per = {k: any(sheet_ok(a, k) for a in s.split("|")) for k in EV}
        ev_any |= any(per.values())
        if all(per.values()):
            ev_disp.append(s)
        else:
            ev_disp.append(f"{s} [{', '.join(k for k, v in per.items() if v) or 'none'} only]")
            miss.append(f"evidence sheet {s} missing or empty for {', '.join(k for k, v in per.items() if not v)}")
    spec_missing = [i for i in r["spec"] if i not in SPEC_BY]
    miss += [f"spec section {i} not found in the spec" for i in spec_missing]
    for hh in r["hi"]:
        if hh not in HI_BY:
            miss.append(f"human-input heading '{hh}' not in {MDD['human_inputs']}")
        elif HI_BY[hh]["open_placeholders"]:
            miss.append(f"human input '{hh}' still open: " + "; ".join(HI_BY[hh]["open_placeholders"][:6]))
    for k in EV:
        for g in EV[k]["gaps"]:
            if g.get("gap_type") in r["gap_types"]:
                miss.append(f"{k}: {g['gap_type']} {g.get('item')}: {norm(g.get('detail'))[:150]}")
    if r["kind"] == "AUTO":
        status = "COVERED"
    elif r["gap"] or not (ev_any or (r["spec"] and not spec_missing)):
        status = "GAP"
    else:
        status = "PARTIAL" if miss else "COVERED"
    excerpt = norm(p.excerpt_200) if isinstance(p.excerpt_200, str) else ""
    prev = f"{p.classification}: " + (excerpt or f"(no paragraph text; {int(p.n_tables)} table(s), {int(p.n_figures)} figure(s))")
    ROWS.append({
        "order": h["order"], "level": h["level"], "template_heading": h["heading"], "parent_heading": h["parent"],
        "template_asks": r["asks"],
        "template_tables": "\n".join(f"T{t}: {TABLES[t]['header']}" + (f"  [items: {TABLES[t]['labels']}]" if TABLES[t]["labels"] else "")
                                     for t in h["tables"]) + ("TOC field" if h["toc_field"] else ""),
        "template_placeholder": "\n".join(h["placeholders"]),
        "spec_sections": "\n".join(f"{i} {SPEC_BY[i]['title']}" if i in SPEC_BY else f"{i} (NOT FOUND)" for i in r["spec"]),
        "evidence_sheets": "\n".join(ev_disp), "human_inputs": "\n".join(r["hi"]),
        "status": status, "what_is_missing": "\n".join(miss), "previous_version": prev,
        "_spec": r["spec"], "_ev": r["ev"], "_hi": r["hi"]})
MAP_DF = pd.DataFrame(ROWS)
MAP_COLS = [c for c in MAP_DF.columns if not c.startswith("_")]
print(f"{len(MAP_DF)} map rows | status {dict(Counter(MAP_DF['status']))} | rules used once: "
      f"{all(RULE_USE[id(r)] == 1 for r in MAP_RULES)} | VARBIN rows {RULE_USE[id(VARBIN)]}")
display(MAP_DF[["order", "level", "template_heading", "status", "spec_sections", "evidence_sheets", "human_inputs", "what_is_missing", "previous_version"]])

In [ ]:
# Units = level-3 spec sections (one dot, e.g. 6.4, D.1) plus chapters/appendices without numbered sub-sections.
# A unit is PLACED if a map row references it, PARTLY if only a sub-section of it is referenced.
REF_SPEC = {i for r in ROWS for i in r["_spec"]}
UNITS = [s for s in SPEC_H if s["id"] and (s["id"].count(".") == 1 or ("." not in s["id"] and not s["has_children"]))]

# proposed destination: exact id first, then chapter id. (destination, section|appendix|omit, evidence sheets that would feed it)
PROPOSE = {
    "0": ("Omit (how to read the specification)", "omit", []),
    "0.4": ("Annexure: glossary of terms and abbreviations", "appendix", []),
    "1.4": ("Modelling Assumptions, Limitations & Weaknesses (open TBC items)", "section", []),
    "2": ("DATA ANALYSIS / Performance Definition", "section", []),
    "3": ("Annexure: implementation conventions and reproducibility (seed, OOT read lock, artefact locations)", "appendix", ["inventory_reports", "registry_fields"]),
    "4": ("Annexure: data-preparation outputs and acceptance results", "appendix", ["data_confirmed_counts_cfg"]),
    "5.3": ("New H3 'Back-test (seasonality)' under Model Performance", "section", []),
    "5.4": ("Model Metrics: OOT split by quarter (Q1/Q2)", "section", ["oot_metrics", "oot_paired_bootstrap"]),
    "6.4": ("New H2 'Distribution stability (PSI)' after 'Trends and Multi collinearity'", "section", ["L3_psi_summary", "L3_red_features", "L3_psi_decisions_file"]),
    "7.4": ("Challenger Models (Model B, EBM)", "section", ["mod_B_shortlist", "mod_B_shape_check", "mod_reg_ebm_shape"]),
    "7.6": ("Scaling (calibration fitted on DEV OOF predictions)", "section", ["mod_oof_metrics", "cal_platt"]),
    "7.7": ("Key dates of Model Life-Cycle (lock date) + Annexure: reproducibility", "section", ["registry_fields"]),
    "8.4": ("New H3 'Back-test (seasonality)' under Model Performance", "section", []),
    "8.5": ("Modelling Assumptions, Limitations & Weaknesses", "section", ["oot_diagnosis"]),
    "9.2": ("Scaling (pre-launch level adjustment / central tendency)", "section", []),
    "10": ("New H2 'Full-data refit' at the end of the model chapter (E16 not run)", "section", []),
    "11": ("New H1 'Stress test (external shock from 2026-03)' before Limitations (E17 not run)", "section", []),
    "12": ("New H1 'Reject inference and policy-change impact'", "section", ["ri_summary|ri_fts_summary", "ri_intake_waterfall|ri_fts_universe_counts"]),
    "13": ("New H1 'Implementation' (API, reason codes, online/offline, shadow run)", "section", ["rc_map_used", "rc_draft_run_summary"]),
    "13.2": ("New H1 'Implementation': reason codes", "section", ["rc_map_used", "rc_draft_run_summary"]),
    "14": ("New H1 'Monitoring and review triggers'", "section", []),
    "16": ("Annexure: review checklists", "appendix", []),
    "16.3": ("Omit (code-agent acceptance) or Annexure: acceptance results", "appendix", []),
    "A": ("Annexure: change log against the first draft specification", "appendix", []),
    "B": ("Annexure: thresholds and parameters", "appendix", []),
    "C": ("Annexure: configuration (config.yaml)", "appendix", []),
    "D": ("Annexure: method reference (reference code, not reproduced)", "appendix", []),
    "E": ("Annexure: implementation task list / module status", "appendix", []),
    "F": ("Annexure: implementation on Databricks", "appendix", []),
    "G": ("Exclusion Rules / Segmentation (file-type definitions)", "section", ["data_filter_waterfall"]),
}
def propose(i):
    return PROPOSE.get(i) or PROPOSE.get(i.split(".")[0]) or ("Annexure (destination to be decided)", "appendix", [])

UNPLACED = []
for s in UNITS:
    i = s["id"]
    if i in REF_SPEC:
        continue
    kids = sorted(x for x in REF_SPEC if x.startswith(i + "."))
    dest, kind, evs = propose(i)
    UNPLACED.append({"spec_id": i, "spec_title": s["title"], "placement": "PARTLY (sub-sections " + ", ".join(kids) + " placed)" if kids else "NOT PLACED",
                     "proposed_destination": dest, "kind": kind,
                     "evidence_available": ", ".join(e for e in evs if any(any(sheet_ok(a, k) for a in e.split("|")) for k in EV)) or "none"})
SPEC_UNPLACED_DF = pd.DataFrame(UNPLACED)

# evidence sheets no template section uses
REF_EV = {a for r in ROWS for s in r["_ev"] for a in s.split("|")}
STRUCT = {"meta", "index", "inventory_reports", "registry_fields", "selection_override", "gaps"}
EV_PREFIX = [("benchmark_", "Challenger Models / Model Results KPI's (comparison with A-Score, AECB)"),
             ("ascore_cmp_", "superseded by benchmark_* sheets (older 05d report)"),
             ("fts_cmp_", "Challenger Models (FTS vs no-FTS)"), ("cut_", "New H2 'Cut-off analysis' after Risk tranches"),
             ("ri_", "New H1 'Reject inference and policy-change impact' (spec 12)"), ("rc_", "New H1 'Implementation': reason codes (spec 13.2)"),
             ("vc_", "Annexure: no-FTS variant vs main build reconciliation"), ("L3_", "New H2 'Distribution stability (PSI)' (spec 6.4)"),
             ("L0_", "Data Quality"), ("L1_", "Information Value"), ("L2_", "Trends and Multi collinearity"), ("L4_", "Correlation Analysis"),
             ("L5_", "Final Model Variables"), ("fun_", "Summary of Variable Reduction"), ("mod_", "Fine Tuning / Challenger Models"),
             ("oot_", "Model Performance"), ("cal_", "Scaling"), ("data_", "Data Analysis / Sample Selection"), ("target_", "Performance Definition")]
EV_UNPLACED_DF = pd.DataFrame([{"sheet": s, **{f"in_{k}": s in EV[k]["sheets"] for k in EV},
                                "proposed_destination": "Annexure: evidence index (structural sheet)" if s in STRUCT else
                                next((d for p_, d in EV_PREFIX if s.startswith(p_)), "Annexure (destination to be decided)")}
                               for s in ALL_SHEETS if s not in REF_EV])
print(f"spec units {len(UNITS)} | referenced by the map {sum(s['id'] in REF_SPEC for s in UNITS)} | no place in the template {len(UNPLACED)} "
      f"({sum(u['placement'] == 'NOT PLACED' for u in UNPLACED)} not placed, {sum(u['placement'] != 'NOT PLACED' for u in UNPLACED)} partly)")
print(f"evidence sheets not used by any section: {len(EV_UNPLACED_DF)} of {len(ALL_SHEETS)}")
display(SPEC_UNPLACED_DF)
display(EV_UNPLACED_DF)

In [ ]:
CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "pass": bool(ok), "detail": str(detail)[:1000]})

check("template = config.mdd.template (clean template from Step 2a)", TPL.endswith("mdd_template_clean.docx") and os.path.isfile(TPL), TPL)
check("one map row per template heading, same order / level / text",
      len(MAP_DF) == len(HEADINGS) and all(r.order == h["order"] and r.level == h["level"] and r.template_heading == h["heading"]
                                          for r, h in zip(MAP_DF.itertuples(), HEADINGS)), f"{len(MAP_DF)} rows vs {len(HEADINGS)} headings")
check("every mapping rule used exactly once (variable H3s share one rule)", all(RULE_USE[id(r)] == 1 for r in MAP_RULES),
      [r["key"] for r in MAP_RULES if RULE_USE[id(r)] != 1])
check("status in COVERED / PARTIAL / GAP", set(MAP_DF["status"]) <= {"COVERED", "PARTIAL", "GAP"}, dict(Counter(MAP_DF["status"])))
check("PARTIAL and GAP rows say what is missing; COVERED rows have nothing missing",
      all((r.status == "COVERED") == (r.what_is_missing == "") for r in MAP_DF.itertuples() if r.template_heading.lower() != "table of contents"))
check("every referenced spec section exists in the spec", REF_SPEC <= set(SPEC_BY), sorted(REF_SPEC - set(SPEC_BY)))
check("every referenced evidence sheet exists in at least one evidence workbook", REF_EV <= set(ALL_SHEETS), sorted(REF_EV - set(ALL_SHEETS)))
ref_hi = {x for r in ROWS for x in r["_hi"]}
check("every referenced human input is a heading of config.mdd.human_inputs", ref_hi <= set(HI_BY), sorted(ref_hi - set(HI_BY)))
check("previous-MDD profile matches the template headings; 'previous version' filled for every row",
      PROF_OK and MAP_DF["previous_version"].str.len().gt(0).all())
check("every spec unit without a place has a proposed destination",
      all(u["proposed_destination"] for u in UNPLACED) and len(UNPLACED) + sum(s["id"] in REF_SPEC for s in UNITS) == len(UNITS))
check("inputs unchanged during the run (sha256)", all(sha256_file(p) == INPUT_SHA[k] for k, p in
      [("config", CFG_PATH), ("template", TPL), ("spec", SPEC), ("human_inputs", HI), ("profile", PROFILE)] + [(f"evidence_{k}", v) for k, v in EV_WB.items()]))

COUNTS = {"template_headings": len(HEADINGS), "template_tables": len(TABLES), "status": dict(Counter(MAP_DF["status"])),
          "spec_units": len(UNITS), "spec_unplaced": len(UNPLACED), "evidence_sheets": len(ALL_SHEETS), "evidence_unplaced": len(EV_UNPLACED_DF)}
META = pd.DataFrame([["module", MODULE], ["run_time_utc", RUN_TIME], ["git_commit", GIT], ["git_branch", BRANCH], ["config_sha256", CFG_SHA],
                     ["template", TPL], ["spec", SPEC], ["human_inputs", HI], ["profile", PROFILE]]
                    + [[f"evidence_{k}", v] for k, v in EV_WB.items()] + [[f"sha256_{k}", v] for k, v in INPUT_SHA.items()]
                    + [["config.mdd duplicate keys (yaml keeps the last)", ", ".join(MDD_DUP_KEYS) or "none"],
                       ["status rule", "GAP: content out of scope / no spec and no evidence; PARTIAL: something listed in what_is_missing; COVERED: spec + evidence + human inputs complete"]],
                    columns=["field", "value"])
RUN_SUMMARY = pd.DataFrame([["module", MODULE], ["run_time_utc", RUN_TIME], ["git_commit", GIT], ["config_hash", CFG_SHA],
                            ["outputs", OUT], ["counts", json.dumps(COUNTS)],
                            ["pre_write_checks", f"{sum(c['pass'] for c in CHECKS)}/{len(CHECKS)} PASS"],
                            ["pass_fail", "PASS" if all(c["pass"] for c in CHECKS) else "FAIL"],
                            ["note", "write-set and re-read checks run after the write; see notebook output"]], columns=["field", "value"])
FRAMES = [("meta", META), ("map", MAP_DF[MAP_COLS]), ("spec_unplaced", SPEC_UNPLACED_DF), ("evidence_unplaced", EV_UNPLACED_DF),
          ("template_headings", TPL_HEAD_DF), ("template_tables", TPL_TBL_DF), ("template_styles", TPL_STYLE_DF),
          ("spec_headings", SPEC_DF), ("evidence_sheets", EV_DF), ("human_inputs", HI_DF), ("checks", pd.DataFrame(CHECKS)),
          ("run_summary", RUN_SUMMARY)]
tmp = os.path.join(tempfile.mkdtemp(), "template_map.xlsx")
with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
    for name, df in FRAMES:
        df.to_excel(xw, sheet_name=name, index=False)
shutil.copyfile(tmp, OUT)

# post-write: re-read the workbook and check the write set
back = pd.read_excel(OUT, sheet_name=None)
post = [("workbook sheets as written, meta first", list(back) == [n for n, _ in FRAMES]),
        ("map sheet re-read equals the map (rows, headings, status)",
         len(back["map"]) == len(MAP_DF) and list(back["map"]["template_heading"]) == list(MAP_DF["template_heading"])
         and list(back["map"]["status"]) == list(MAP_DF["status"]))]
SNAP_AFTER = snapshot(WATCH)
changed = sorted(p for p in set(SNAP_BEFORE) | set(SNAP_AFTER) if SNAP_BEFORE.get(p) != SNAP_AFTER.get(p))
post.append(("only template_map.xlsx written (before/after snapshot of output folder, docs/, config/, spec folder)", changed == [OUT]))
for c in CHECKS:
    print(("PASS  " if c["pass"] else "FAIL  ") + c["check"] + ("" if c["pass"] else f" | {c['detail']}"))
for n, ok in post:
    print(("PASS  " if ok else "FAIL  ") + n)
if changed != [OUT]:
    print("changed:", changed[:20])
print(f"wrote {OUT} ({os.path.getsize(OUT):,} bytes, {len(FRAMES)} sheets) | {COUNTS}")
for c in CHECKS:
    assert c["pass"], f"acceptance failed: {c['check']} | {c['detail']}"
for n, ok in post:
    assert ok, f"acceptance failed: {n}"
print("All acceptance checks PASS")